# Cuaderno 9 - Ordenar, buscar extremos y construir rankings

## Antes de comenzar

En los últimos cuadernos aprendimos a seleccionar columnas, filtrar filas, combinar condiciones y construir nuevas variables.

Ahora vamos a trabajar con el **orden de las filas**.

Exploraremos `sort()`, distintos criterios de ordenamiento, máximos y mínimos, selección de los valores extremos y rankings mediante `rank()`.

Seguiremos utilizando el pequeño conjunto de ventas de la tienda escolar, lo que permitirá interpretar cada cambio de orden con facilidad.


## 1. Preparar el entorno y los datos

Instalamos Polars, importamos la biblioteca y reconstruimos el DataFrame habitual.


In [57]:
!pip install -q polars

In [58]:
import polars as pl

In [59]:
datos = {
    "producto": ["Cuaderno", "Lapicera", "Mochila", "Calculadora", "Regla", "Carpeta"],
    "categoria": ["Librería", "Librería", "Accesorios", "Tecnología", "Librería", "Librería"],
    "precio": [3500, 1200, 28500, 18500, 900, 4200],
    "cantidad": [4, 10, 2, 3, 6, 5],
}

df = pl.DataFrame(datos)
df

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


El DataFrame ejecutado tiene `shape: (6, 4)` y conserva el orden en que fue creado: `Cuaderno`, `Lapicera`, `Mochila`, `Calculadora`, `Regla` y `Carpeta`.

Ese orden todavía no expresa ningún criterio analítico; simplemente reproduce la secuencia original de los registros.


## 2. Orden ascendente

Comenzamos ordenando los productos por `precio`, de menor a mayor.


In [60]:
df.sort("precio")

producto,categoria,precio,cantidad
str,str,i64,i64
"""Regla""","""Librería""",900,6
"""Lapicera""","""Librería""",1200,10
"""Cuaderno""","""Librería""",3500,4
"""Carpeta""","""Librería""",4200,5
"""Calculadora""","""Tecnología""",18500,3
"""Mochila""","""Accesorios""",28500,2


La salida mantiene las seis filas y cuatro columnas, pero ahora aparecen ordenadas por `precio` de menor a mayor:

```text
Regla        900
Lapicera    1200
Cuaderno    3500
Carpeta     4200
Calculadora 18500
Mochila     28500
```

`sort()` reorganiza filas completas. Cada producto sigue asociado a su categoría, precio y cantidad; solamente cambia su posición dentro del DataFrame.


## 3. Orden descendente

Para colocar primero los precios más altos utilizamos `descending=True`.


In [61]:
df.sort(
    "precio",
    descending=True,
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Carpeta""","""Librería""",4200,5
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Regla""","""Librería""",900,6


El criterio se invierte por completo. La salida comienza con `Mochila` (`28500`) y `Calculadora` (`18500`) y termina con `Lapicera` (`1200`) y `Regla` (`900`).

`descending=True` convierte el orden ascendente predeterminado en un orden de mayor a menor.


## 4. El DataFrame original no cambia

Guardamos una versión ordenada y volvemos a observar `df`.


In [62]:
ordenado = df.sort(
    "precio",
    descending=True,
)

ordenado

producto,categoria,precio,cantidad
str,str,i64,i64
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Carpeta""","""Librería""",4200,5
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Regla""","""Librería""",900,6


In [63]:
df

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


`ordenado` muestra las filas desde el precio más alto hasta el más bajo. Al observar inmediatamente después `df`, reaparece el orden original: `Cuaderno`, `Lapicera`, `Mochila`, `Calculadora`, `Regla` y `Carpeta`.

Esto confirma que `sort()` produjo un nuevo DataFrame y no reorganizó `df` en forma permanente.


## 5. Ordenar por texto

También podemos ordenar utilizando una columna de texto.


In [64]:
df.sort("producto")

producto,categoria,precio,cantidad
str,str,i64,i64
"""Calculadora""","""Tecnología""",18500,3
"""Carpeta""","""Librería""",4200,5
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Regla""","""Librería""",900,6


La salida queda ordenada alfabéticamente por `producto`: `Calculadora`, `Carpeta`, `Cuaderno`, `Lapicera`, `Mochila` y `Regla`.

El mecanismo es el mismo que con una columna numérica; cambia únicamente el criterio utilizado para comparar los valores.


## 6. Ordenar por varios criterios

Ahora ordenamos primero por `categoria` y, dentro de cada categoría, por `precio`.


In [65]:
df.sort(
    ["categoria", "precio"]
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Mochila""","""Accesorios""",28500,2
"""Regla""","""Librería""",900,6
"""Lapicera""","""Librería""",1200,10
"""Cuaderno""","""Librería""",3500,4
"""Carpeta""","""Librería""",4200,5
"""Calculadora""","""Tecnología""",18500,3


La salida muestra primero `Accesorios`, luego los cuatro registros de `Librería` y finalmente `Tecnología`.

Dentro de `Librería`, el segundo criterio ordena por precio de menor a mayor: `Regla` (`900`), `Lapicera` (`1200`), `Cuaderno` (`3500`) y `Carpeta` (`4200`).

Esto permite observar la jerarquía de criterios: `categoria` decide primero y `precio` resuelve el orden dentro de cada categoría.


## 7. Un sentido diferente para cada criterio

Podemos mantener las categorías en orden ascendente y ordenar los precios en sentido descendente dentro de cada una.


In [66]:
df.sort(
    ["categoria", "precio"],
    descending=[False, True],
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Mochila""","""Accesorios""",28500,2
"""Carpeta""","""Librería""",4200,5
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Regla""","""Librería""",900,6
"""Calculadora""","""Tecnología""",18500,3


La salida mantiene `categoria` en sentido ascendente, pero dentro de `Librería` los precios aparecen ahora de mayor a menor: `Carpeta`, `Cuaderno`, `Lapicera` y `Regla`.

La lista:

```text
[False, True]
```

se corresponde con:

```text
categoria → ascendente
precio    → descendente
```


## 8. Filtrar y después ordenar

Conservamos solamente los productos de Librería y los organizamos desde el más caro hasta el más barato.


In [67]:
df.filter(
    pl.col("categoria") == "Librería"
).sort(
    "precio",
    descending=True,
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Carpeta""","""Librería""",4200,5
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Regla""","""Librería""",900,6


El filtro deja cuatro filas, todas de `Librería`, y `sort()` las reorganiza por precio descendente:

```text
Carpeta     4200
Cuaderno    3500
Lapicera    1200
Regla        900
```

La cadena muestra claramente dos etapas distintas: `filter()` decide qué filas permanecen y `sort()` establece en qué orden se presentan.


## 9. Ordenar por una columna calculada

Creamos `importe` como `precio × cantidad` y utilizamos esa nueva columna como criterio de orden.


In [68]:
df.with_columns(
    importe=pl.col("precio") * pl.col("cantidad")
).sort(
    "importe",
    descending=True,
)

producto,categoria,precio,cantidad,importe
str,str,i64,i64,i64
"""Mochila""","""Accesorios""",28500,2,57000
"""Calculadora""","""Tecnología""",18500,3,55500
"""Carpeta""","""Librería""",4200,5,21000
"""Cuaderno""","""Librería""",3500,4,14000
"""Lapicera""","""Librería""",1200,10,12000
"""Regla""","""Librería""",900,6,5400


La nueva columna `importe` contiene `57000`, `55500`, `21000`, `14000`, `12000` y `5400` una vez ordenada de mayor a menor.

En este conjunto el orden por importe coincide con el orden por precio, pero eso ocurre por los valores concretos de `cantidad`. El criterio analítico es distinto: `importe` combina `precio × cantidad`, por lo que en otros datos ambos rankings podrían diferir.


## 10. Construir una tabla enfocada

Podemos seleccionar solamente las variables que necesitamos para interpretar ese orden.


In [69]:
df.with_columns(
    importe=pl.col("precio") * pl.col("cantidad")
).sort(
    "importe",
    descending=True,
).select(
    "producto",
    "precio",
    "cantidad",
    "importe",
)

producto,precio,cantidad,importe
str,i64,i64,i64
"""Mochila""",28500,2,57000
"""Calculadora""",18500,3,55500
"""Carpeta""",4200,5,21000
"""Cuaderno""",3500,4,14000
"""Lapicera""",1200,10,12000
"""Regla""",900,6,5400


La salida tiene `shape: (6, 4)` y contiene solamente `producto`, `precio`, `cantidad` e `importe`, ordenados por este último de mayor a menor.

La cadena `with_columns() → sort() → select()` construye así una tabla enfocada específicamente en comparar el valor total de cada registro.


## 11. Localizar el precio máximo

Si ordenamos de mayor a menor y conservamos la primera fila, obtenemos el registro asociado al precio máximo.


In [70]:
df.sort(
    "precio",
    descending=True,
).head(1)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Mochila""","""Accesorios""",28500,2


La salida tiene una única fila: `Mochila`, con precio `28500`, categoría `Accesorios` y cantidad `2`.

Esta estrategia devuelve el registro completo asociado al máximo, no solamente el número `28500`.


## 12. Localizar el precio mínimo

Invertimos el criterio.


In [71]:
df.sort("precio").head(1)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Regla""","""Librería""",900,6


La salida tiene una única fila: `Regla`, con precio `900`.

Al ordenar de menor a mayor y conservar `head(1)`, obtenemos el registro completo asociado al valor mínimo.


## 13. Obtener varios extremos

Podemos conservar más de una posición. Buscamos primero los tres productos más caros.


In [72]:
df.sort(
    "precio",
    descending=True,
).head(3)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Carpeta""","""Librería""",4200,5


La salida contiene los tres productos de mayor precio: `Mochila` (`28500`), `Calculadora` (`18500`) y `Carpeta` (`4200`).

La combinación `sort(..., descending=True).head(3)` hace explícitos los dos pasos: ordenar y conservar las primeras tres posiciones.


## 14. `top_k()` y `bottom_k()`

Polars también ofrece operaciones específicas para seleccionar registros asociados a los valores extremos.


In [73]:
df.top_k(
    3,
    by="precio",
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Carpeta""","""Librería""",4200,5


En esta ejecución, `top_k(3, by="precio")` devuelve `Mochila`, `Calculadora` y `Carpeta`, que son efectivamente los tres precios más altos.

Conviene mantener una distinción importante: `top_k()` selecciona las filas asociadas a los valores extremos, pero no debemos depender de él para imponer un orden de presentación garantizado. Si ese orden importa, podemos aplicar `sort()` después.


In [74]:
df.bottom_k(
    3,
    by="precio",
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Regla""","""Librería""",900,6
"""Lapicera""","""Librería""",1200,10
"""Cuaderno""","""Librería""",3500,4


`bottom_k(3, by="precio")` devuelve `Regla`, `Lapicera` y `Cuaderno`, correspondientes a los tres precios más bajos.

En esta ejecución aparecen además de menor a mayor, pero, del mismo modo que con `top_k()`, si necesitamos garantizar un orden concreto conviene expresarlo explícitamente con `sort()`.


## 15. El valor máximo no es lo mismo que su fila

Podemos obtener directamente el máximo y el mínimo de una columna.


In [75]:
df.select(
    pl.col("precio").max().alias("precio_maximo"),
    pl.col("precio").min().alias("precio_minimo"),
)

precio_maximo,precio_minimo
i64,i64
28500,900


La salida se reduce a una sola fila con dos valores:

```text
precio_maximo = 28500
precio_minimo = 900
```

Aquí ya no obtenemos los registros completos, sino dos resúmenes escalares de la columna `precio`. Esta es una pregunta diferente de localizar qué producto contiene cada extremo.


## 16. Crear un ranking

Ahora queremos conservar el orden original del DataFrame y agregar una columna que indique la posición de cada producto según su precio.

El precio más alto recibirá la posición `1`.


In [76]:
df.with_columns(
    ranking_precio=(
        pl.col("precio").rank(
            method="ordinal",
            descending=True,
        )
    )
)

producto,categoria,precio,cantidad,ranking_precio
str,str,i64,i64,u32
"""Cuaderno""","""Librería""",3500,4,4
"""Lapicera""","""Librería""",1200,10,5
"""Mochila""","""Accesorios""",28500,2,1
"""Calculadora""","""Tecnología""",18500,3,2
"""Regla""","""Librería""",900,6,6
"""Carpeta""","""Librería""",4200,5,3


La nueva columna `ranking_precio` es de tipo `UInt32` y asigna estas posiciones:

```text
Mochila       1
Calculadora   2
Carpeta       3
Cuaderno      4
Lapicera      5
Regla         6
```

Sin embargo, las filas siguen apareciendo en el orden original. `rank()` calcula una posición para cada registro, pero no reorganiza el DataFrame.


## 17. Ordenar según el ranking

Podemos utilizar la nueva columna para mostrar la clasificación en orden.


In [77]:
df.with_columns(
    ranking_precio=(
        pl.col("precio").rank(
            method="ordinal",
            descending=True,
        )
    )
).sort("ranking_precio")

producto,categoria,precio,cantidad,ranking_precio
str,str,i64,i64,u32
"""Mochila""","""Accesorios""",28500,2,1
"""Calculadora""","""Tecnología""",18500,3,2
"""Carpeta""","""Librería""",4200,5,3
"""Cuaderno""","""Librería""",3500,4,4
"""Lapicera""","""Librería""",1200,10,5
"""Regla""","""Librería""",900,6,6


Después de ordenar por `ranking_precio`, las filas aparecen efectivamente en el orden de la clasificación: `Mochila`, `Calculadora`, `Carpeta`, `Cuaderno`, `Lapicera` y `Regla`.

La diferencia queda completamente visible:

```text
rank() → calcula posiciones
sort() → reorganiza filas
```


## 18. Observar qué ocurre con los empates

Nuestro DataFrame original no tiene precios repetidos, así que creamos una tabla pequeña solamente para observar cómo distintos métodos de ranking tratan los empates.


In [78]:
ejemplo = pl.DataFrame({
    "producto": ["A", "B", "C", "D"],
    "ventas": [100, 80, 80, 50],
})

ejemplo

producto,ventas
str,i64
"""A""",100
"""B""",80
"""C""",80
"""D""",50


Comenzamos con `ordinal`.


In [79]:
ejemplo.with_columns(
    ranking=pl.col("ventas").rank(
        method="ordinal",
        descending=True,
    )
)

producto,ventas,ranking
str,i64,u32
"""A""",100,1
"""B""",80,2
"""C""",80,3
"""D""",50,4


Con `method="ordinal"`, los cuatro registros reciben posiciones distintas:

```text
A → 1
B → 2
C → 3
D → 4
```

Aunque `B` y `C` tienen el mismo valor (`80`), el método ordinal asigna una posición diferente a cada fila.


Probamos ahora `dense`.


In [80]:
ejemplo.with_columns(
    ranking=pl.col("ventas").rank(
        method="dense",
        descending=True,
    )
)

producto,ventas,ranking
str,i64,u32
"""A""",100,1
"""B""",80,2
"""C""",80,2
"""D""",50,3


Con `method="dense"`, el empate se refleja directamente en el ranking:

```text
A → 1
B → 2
C → 2
D → 3
```

`B` y `C` comparten el segundo lugar y el siguiente valor recibe la posición consecutiva `3`.


Finalmente observamos `min`.


In [81]:
ejemplo.with_columns(
    ranking=pl.col("ventas").rank(
        method="min",
        descending=True,
    )
)

producto,ventas,ranking
str,i64,u32
"""A""",100,1
"""B""",80,2
"""C""",80,2
"""D""",50,4


Con `method="min"` obtenemos:

```text
A → 1
B → 2
C → 2
D → 4
```

Los valores empatados reciben la menor posición que les correspondería y la posición `3` queda sin utilizar. La salida permite comparar de forma muy clara este comportamiento con `dense`.


## 19. Ranking de una columna calculada

Creamos primero `importe` y después calculamos su ranking. Como la segunda expresión depende de una columna recién creada, utilizamos dos llamadas a `with_columns()`.


In [82]:
df.with_columns(
    importe=pl.col("precio") * pl.col("cantidad")
).with_columns(
    ranking_importe=(
        pl.col("importe").rank(
            method="ordinal",
            descending=True,
        )
    )
).sort("ranking_importe")

producto,categoria,precio,cantidad,importe,ranking_importe
str,str,i64,i64,i64,u32
"""Mochila""","""Accesorios""",28500,2,57000,1
"""Calculadora""","""Tecnología""",18500,3,55500,2
"""Carpeta""","""Librería""",4200,5,21000,3
"""Cuaderno""","""Librería""",3500,4,14000,4
"""Lapicera""","""Librería""",1200,10,12000,5
"""Regla""","""Librería""",900,6,5400,6


La salida ordenada por `ranking_importe` muestra:

```text
Mochila       57000 → 1
Calculadora   55500 → 2
Carpeta       21000 → 3
Cuaderno      14000 → 4
Lapicera      12000 → 5
Regla          5400 → 6
```

La secuencia queda así completamente visible:

1. calcular `importe`;
2. asignarle un ranking;
3. ordenar según ese ranking.


## 20. Una consulta integrada

Terminamos reuniendo varias herramientas de los capítulos anteriores.

Conservamos productos cuyo precio supera `1000`, calculamos su importe, clasificamos ese importe, ordenamos de mayor a menor y seleccionamos las columnas relevantes.


In [83]:
df.filter(
    pl.col("precio") > 1000
).with_columns(
    importe=pl.col("precio") * pl.col("cantidad")
).with_columns(
    nivel_importe=(
        pl.when(
            pl.col("importe") > 50000
        )
        .then(pl.lit("Alto"))
        .otherwise(pl.lit("Normal"))
    )
).sort(
    "importe",
    descending=True,
).select(
    "producto",
    "precio",
    "cantidad",
    "importe",
    "nivel_importe",
)

producto,precio,cantidad,importe,nivel_importe
str,i64,i64,i64,str
"""Mochila""",28500,2,57000,"""Alto"""
"""Calculadora""",18500,3,55500,"""Alto"""
"""Carpeta""",4200,5,21000,"""Normal"""
"""Cuaderno""",3500,4,14000,"""Normal"""
"""Lapicera""",1200,10,12000,"""Normal"""


La salida final tiene `shape: (5, 5)`. `Regla` queda fuera porque su precio (`900`) no supera `1000`.

Entre las cinco filas restantes, `Mochila` (`57000`) y `Calculadora` (`55500`) reciben `nivel_importe = "Alto"`. `Carpeta`, `Cuaderno` y `Lapicera` quedan como `"Normal"`.

Además, el resultado aparece ordenado por `importe` de mayor a menor. La cadena reúne de manera concreta buena parte de la caja de herramientas construida hasta aquí:

```text
filter()
with_columns()
when() / then() / otherwise()
sort()
select()
```


## 21. El DataFrame original sigue intacto

Volvemos a mostrar `df`.


In [84]:
df

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


Al volver a mostrar `df`, reaparecen las seis filas, las cuatro columnas originales y el orden inicial.

Ninguno de los ordenamientos, rankings ni transformaciones anteriores modificó el DataFrame de partida; todos construyeron nuevos resultados.


## 22. Qué conviene recordar

En este cuaderno incorporamos herramientas para trabajar con el orden y la posición relativa de los registros.

Vimos que:

- `sort()` reorganiza filas completas;
- podemos ordenar de forma ascendente o descendente;
- pueden utilizarse varios criterios de orden;
- es posible ordenar después de filtrar o según una columna calculada;
- `sort().head()` permite recuperar registros extremos;
- `max()` y `min()` devuelven el valor extremo, no necesariamente la fila que lo contiene;
- `top_k()` y `bottom_k()` seleccionan registros asociados a extremos;
- `rank()` calcula posiciones sin reorganizar automáticamente el DataFrame;
- distintos métodos de ranking resuelven los empates de diferentes maneras.

Con esto cerramos una primera etapa dedicada al trabajo cotidiano con expresiones y transformaciones.


## Próximo paso

Hasta ahora trabajamos con un DataFrame deliberadamente pequeño y limpio.

En el próximo capítulo cambiaremos el foco. Comenzaremos a mirar con mayor atención **qué tipos de datos contiene realmente cada columna**, cómo se representa el esquema y qué ocurre cuando los datos no llegan con el tipo que necesitamos.

Ese cambio abre la siguiente parte del recorrido: la preparación y limpieza de datos.
